# Etapa 2 — Modelagem e Avaliação

**Objetivo:** comparar três famílias de modelos (Regressão Logística, Random Forest e MLP) com
validação cruzada estratificada, ajustar o limiar de decisão e escolher o modelo campeão para
servir via API.

**Ponto de partida (Etapa 1 — `01_eda.ipynb`):**
- Baseline de Regressão Logística no teste: recall 0,57 · precision 0,64 · F1 0,60 · AUC-ROC 0,849.
- Métrica principal: **recall** (objetivo 0,70–0,80); secundária: F1.

**Regra desta etapa:** o conjunto de teste é usado **uma única vez**, na avaliação final do modelo
campeão. Toda comparação, tuning e escolha de limiar é feita com validação cruzada no treino.

## 1. Preparação dos dados

In [1]:
# Imports e semente global (reprodutibilidade)
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

In [2]:
# Carga dos dados + correção de Total Charges (mesmo tratamento do 01_eda)
df = pd.read_csv("../data/Telco_customer_churn.csv", sep=";", decimal=",")

df["Total Charges"] = pd.to_numeric(
    df["Total Charges"].str.replace(",", "."), errors="coerce"
).fillna(0)

df.shape, df["Total Charges"].dtype

((7043, 33), dtype('float64'))

In [3]:
# Features e target — mesmas listas do baseline (sem colunas com vazamento:
# Churn Score, CLTV, Churn Reason etc.)
numeric_cols = ["Tenure Months", "Monthly Charges", "Total Charges"]

categorical_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method",
]

X = df[numeric_cols + categorical_cols]
y = df["Churn Value"]

# Split 80/20 estratificado — idêntico ao da Etapa 1 para manter comparabilidade
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)

X_train.shape, X_test.shape

((5634, 19), (1409, 19))

In [4]:
# Conferência: proporção de churn preservada nos dois conjuntos
pd.DataFrame({
    "treino": y_train.value_counts(normalize=True),
    "teste": y_test.value_counts(normalize=True),
}).round(3)

,treino,teste
Churn Value,,
0,0.735,0.735
1,0.265,0.265


## 2. Validação cruzada e Regressão Logística

In [5]:
# Validação cruzada estratificada (5 folds)
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Conferência: tamanho de cada fold e proporção de churn
for i, (idx_treino, idx_valid) in enumerate(cv.split(X_train, y_train), start=1):
    taxa_churn = y_train.iloc[idx_valid].mean()
    print(f"Fold {i}: treino={len(idx_treino)}  validação={len(idx_valid)}  churn={taxa_churn:.3f}")

Fold 1: treino=4507  validação=1127  churn=0.265
Fold 2: treino=4507  validação=1127  churn=0.265
Fold 3: treino=4507  validação=1127  churn=0.265
Fold 4: treino=4507  validação=1127  churn=0.265
Fold 5: treino=4508  validação=1126  churn=0.266


In [6]:
# Regressão Logística — mesma configuração do baseline da Etapa 1
# (sem class_weight="balanced": a segmentação por risco precisa de probabilidades calibradas)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor_escalado = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

modelo_logreg = Pipeline([
    ("preprocessor", preprocessor_escalado),
    ("classifier", LogisticRegression(max_iter=1000, random_state=SEED)),
])

In [7]:
# Avaliação com validação cruzada — 4 métricas por fold
from sklearn.model_selection import cross_validate

METRICAS = ["roc_auc", "recall", "precision", "f1"]

resultado_logreg = cross_validate(
    modelo_logreg, X_train, y_train,
    cv=cv, scoring=METRICAS,
)

# Nota de cada fold (para ver a variação entre as rodadas)
pd.DataFrame({m: resultado_logreg[f"test_{m}"] for m in METRICAS}).round(3)

,roc_auc,recall,precision,f1
0,0.863,0.575,0.669,0.619
1,0.838,0.532,0.624,0.574
2,0.852,0.562,0.675,0.613
3,0.873,0.659,0.670,0.664
4,0.869,0.582,0.696,0.634


In [8]:
# Registro dos experimentos: média ± desvio-padrão por métrica
NOMES_METRICAS = {"roc_auc": "AUC-ROC", "recall": "Recall", "precision": "Precision", "f1": "F1"}

experimentos = []


def formatar_params(params):
    """Converte {'classifier__C': 100} em 'C=100' para leitura na tabela."""
    return ", ".join(f"{k.replace('classifier__', '')}={v}" for k, v in params.items())


def registrar_experimento(nome, resultado_cv, hiperparametros="padrão"):
    linha = {"Modelo": nome}
    for m in METRICAS:
        notas = resultado_cv[f"test_{m}"]
        linha[NOMES_METRICAS[m]] = f"{notas.mean():.3f} ± {notas.std():.3f}"
    linha["Tempo treino (s)"] = f"{resultado_cv['fit_time'].mean():.2f}"
    linha["Hiperparâmetros"] = hiperparametros
    experimentos.append(linha)


def tabela_experimentos():
    return pd.DataFrame(experimentos).set_index("Modelo")


def salvar_experimentos(caminho="../docs/experimentos.md"):
    tabela = pd.DataFrame(experimentos)
    cabecalho = "| " + " | ".join(tabela.columns) + " |"
    separador = "|" + "---|" * len(tabela.columns)
    linhas = ["| " + " | ".join(map(str, l)) + " |" for l in tabela.values]
    with open(caminho, "w", encoding="utf-8") as f:
        f.write("# Registro de Experimentos — Etapa 2\n\n")
        f.write("Validação cruzada estratificada (5 folds) no conjunto de treino. ")
        f.write("Valores: média ± desvio-padrão. Métrica de seleção: AUC-ROC.\n\n")
        f.write("\n".join([cabecalho, separador] + linhas) + "\n")


registrar_experimento("Regressão Logística (baseline)", resultado_logreg,
                      "padrão (C=1, L2), sem class_weight")
salvar_experimentos()
tabela_experimentos()

,AUC-ROC,Recall,Precision,F1,Tempo treino (s),Hiperparâmetros
Modelo,,,,,,
Regressão Logística (baseline),0.859 ± 0.013,0.582 ± 0.042,0.667 ± 0.024,0.621 ± 0.029,0.03,"padrão (C=1, L2), sem class_weight"


In [9]:
# Regressão Logística — ajuste de hiperparâmetros (força e tipo de regularização)
from sklearn.model_selection import GridSearchCV

grade_logreg = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    # l1_ratio: 0 = regularização L2 (reduz todos os pesos), 1 = L1 (pode zerar variáveis)
    "classifier__l1_ratio": [0, 1],
}

modelo_logreg_busca = Pipeline([
    ("preprocessor", preprocessor_escalado),
    ("classifier", LogisticRegression(solver="liblinear", max_iter=1000, random_state=SEED)),
])

busca_logreg = GridSearchCV(
    modelo_logreg_busca, grade_logreg,
    cv=cv, scoring="roc_auc", n_jobs=-1,
)
busca_logreg.fit(X_train, y_train)

print("Melhor combinação:", formatar_params(busca_logreg.best_params_))
print(f"Melhor AUC (CV): {busca_logreg.best_score_:.3f}")

# Reavalia a melhor combinação com as 4 métricas, no mesmo esquema de CV, para registrar
resultado_logreg_ajustado = cross_validate(
    busca_logreg.best_estimator_, X_train, y_train,
    cv=cv, scoring=METRICAS,
)
registrar_experimento("Regressão Logística (ajustada)", resultado_logreg_ajustado,
                      "GridSearchCV: " + formatar_params(busca_logreg.best_params_))
salvar_experimentos()
tabela_experimentos()

Melhor combinação: C=100, l1_ratio=0
Melhor AUC (CV): 0.859


,AUC-ROC,Recall,Precision,F1,Tempo treino (s),Hiperparâmetros
Modelo,,,,,,
Regressão Logística (baseline),0.859 ± 0.013,0.582 ± 0.042,0.667 ± 0.024,0.621 ± 0.029,0.03,"padrão (C=1, L2), sem class_weight"
Regressão Logística (ajustada),0.859 ± 0.013,0.585 ± 0.043,0.665 ± 0.025,0.621 ± 0.031,0.03,"GridSearchCV: C=100, l1_ratio=0"


## 3. Random Forest

In [10]:
# Random Forest — primeira rodada com hiperparâmetros padrão (referência)
from sklearn.ensemble import RandomForestClassifier

# Árvores não precisam de escalonamento: numéricas passam sem transformação
preprocessor_arvore = ColumnTransformer([
    ("num", "passthrough", numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

modelo_rf = Pipeline([
    ("preprocessor", preprocessor_arvore),
    ("classifier", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
])

resultado_rf = cross_validate(
    modelo_rf, X_train, y_train,
    cv=cv, scoring=METRICAS,
)

registrar_experimento("Random Forest (padrão)", resultado_rf,
                      "padrão do sklearn (100 árvores, sem limite de profundidade)")
salvar_experimentos()
tabela_experimentos()

,AUC-ROC,Recall,Precision,F1,Tempo treino (s),Hiperparâmetros
Modelo,,,,,,
Regressão Logística (baseline),0.859 ± 0.013,0.582 ± 0.042,0.667 ± 0.024,0.621 ± 0.029,0.03,"padrão (C=1, L2), sem class_weight"
Regressão Logística (ajustada),0.859 ± 0.013,0.585 ± 0.043,0.665 ± 0.025,0.621 ± 0.031,0.03,"GridSearchCV: C=100, l1_ratio=0"
Random Forest (padrão),0.837 ± 0.013,0.513 ± 0.029,0.652 ± 0.027,0.574 ± 0.028,0.16,"padrão do sklearn (100 árvores, sem limite de ..."


In [11]:
# Random Forest — ajuste de hiperparâmetros (busca aleatória, seleção por AUC-ROC)
from sklearn.model_selection import RandomizedSearchCV

grade_rf = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [4, 6, 8, 10, 12, None],
    "classifier__min_samples_leaf": [1, 5, 10, 20, 40],
    "classifier__max_features": ["sqrt", 0.3, 0.5],
}

busca_rf = RandomizedSearchCV(
    modelo_rf, grade_rf,
    n_iter=30, cv=cv, scoring="roc_auc",
    random_state=SEED, n_jobs=-1,
)
busca_rf.fit(X_train, y_train)

print("Melhor combinação:", formatar_params(busca_rf.best_params_))
print(f"Melhor AUC (CV): {busca_rf.best_score_:.3f}")

# Reavalia a melhor combinação com as 4 métricas, no mesmo esquema de CV, para registrar
resultado_rf_ajustado = cross_validate(
    busca_rf.best_estimator_, X_train, y_train,
    cv=cv, scoring=METRICAS,
)
registrar_experimento("Random Forest (ajustado)", resultado_rf_ajustado,
                      "RandomizedSearchCV (30 combinações): " + formatar_params(busca_rf.best_params_))
salvar_experimentos()
tabela_experimentos()

Melhor combinação: n_estimators=300, min_samples_leaf=10, max_features=0.5, max_depth=6
Melhor AUC (CV): 0.864


,AUC-ROC,Recall,Precision,F1,Tempo treino (s),Hiperparâmetros
Modelo,,,,,,
Regressão Logística (baseline),0.859 ± 0.013,0.582 ± 0.042,0.667 ± 0.024,0.621 ± 0.029,0.03,"padrão (C=1, L2), sem class_weight"
Regressão Logística (ajustada),0.859 ± 0.013,0.585 ± 0.043,0.665 ± 0.025,0.621 ± 0.031,0.03,"GridSearchCV: C=100, l1_ratio=0"
Random Forest (padrão),0.837 ± 0.013,0.513 ± 0.029,0.652 ± 0.027,0.574 ± 0.028,0.16,"padrão do sklearn (100 árvores, sem limite de ..."
Random Forest (ajustado),0.864 ± 0.010,0.551 ± 0.022,0.673 ± 0.031,0.605 ± 0.020,0.57,RandomizedSearchCV (30 combinações): n_estimat...
